# 27. Calibration and performance diagnosis
A classifier can rank cases well but assign misleading probabilities. Calibration checks whether predicted probabilities match observed frequencies for comparable predictions. Learning curves compare performance at different training sizes. Validation curves compare model settings. Diagnose a problem before prescribing more data or a more complex model.

Use development partitions for calibration and diagnostics. CalibratedClassifierCV fits calibration without allowing calibration examples to directly train the corresponding base predictor. Inspect reliability curves and probability losses. Examine subgroup error counts and sample sizes; a small subgroup score is noisy. Explain possible harms and measurement limitations without treating group differences as causal findings.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
Xt, Xv, yt, yv = train_test_split(X_dev, y_dev, stratify=y_dev, random_state=7) # Reserve development validation examples.
calibrated = CalibratedClassifierCV(model, method="sigmoid", cv=3) # Calibrate through internal training folds.
calibrated.fit(Xt, yt) # Fit without final-test information.
p = calibrated.predict_proba(Xv)[:, 1] # Probabilities for class 1.
print("Validation Brier loss:", round(brier_score_loss(yv, p), 4)) # Evaluate squared probability error.
print("Probability range:", round(p.min(), 3), round(p.max(), 3)) # Inspect valid range.
assert ((p >= 0) & (p <= 1)).all() # Probabilities must be bounded.

## Your turn
Plot a calibration curve and a learning curve. Write a hypothesis about the main limitation, then name evidence that could disprove it.

Try the task before opening the solution on the course website.